# Sentiment analysis

In [25]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

## Data

In [ ]:
fn='./data/rt-polarity.neg'

with open(fn, "r",encoding='utf-8', errors='ignore') as f:
    content = f.read()
texts_neg=  content.splitlines()
print ('len of texts_neg = {:,}'.format (len(texts_neg)))
for review in texts_neg[:5]:
    print ( '\n', review)

len of texts_neg = 5,331

 simplistic , silly and tedious . 

 it's so laddish and juvenile , only teenage boys could possibly find it funny . 

 exploitative and largely devoid of the depth or sophistication that would make watching such a graphic treatment of the crimes bearable . 

 [garbus] discards the potential for pathological study , exhuming instead , the skewed melodrama of the circumstantial situation . 

 a visually flashy but narratively opaque and emotionally vapid exercise in style and mystification . 


In [4]:
fn='./data/rt-polarity.pos'

with open(fn, "r",encoding='utf-8', errors='ignore') as f:
    content = f.read()
texts_pos=  content.splitlines()
print ('len of texts_pos = {:,}'.format (len(texts_pos)))
for review in texts_pos[:5]:
    print ('\n', review)

len of texts_pos = 5,331

 the rock is destined to be the 21st century's new " conan " and that he's going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal . 

 the gorgeously elaborate continuation of " the lord of the rings " trilogy is so huge that a column of words cannot adequately describe co-writer/director peter jackson's expanded vision of j . r . r . tolkien's middle-earth . 

 effective but too-tepid biopic

 if you sometimes like to go to the movies to have fun , wasabi is a good place to start . 

 emerges as something rare , an issue movie that's so honest and keenly observed that it doesn't feel like one . 


## Concatenate dataframes

In [8]:
df_neg = pd.DataFrame({'text': texts_neg, 'label': 0})
df_pos = pd.DataFrame({'text': texts_pos, 'label': 1})

df = pd.concat([df_neg, df_pos], ignore_index=True)

print (df_neg.shape)
print (df_pos.shape)
print (df.shape)

(5331, 2)
(5331, 2)
(10662, 2)


## Train-test split

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], 
    test_size=0.2, 
    random_state=42, 
    stratify=df['label']
)

X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

## TF-IDF + Logistic Regression

In [13]:
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=5)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

lr_tfidf = LogisticRegression(max_iter=1000, random_state=42)
lr_tfidf.fit(X_train_tfidf, y_train)

preds_tfidf = lr_tfidf.predict(X_test_tfidf)

metrics_tfidf = {
    'Accuracy': accuracy_score(y_test, preds_tfidf),
    'Precision': precision_score(y_test, preds_tfidf),
    'Recall': recall_score(y_test, preds_tfidf),
    'F1-Score': f1_score(y_test, preds_tfidf)
}

for metric_name, value in metrics_tfidf.items():
    print(f"- {metric_name}: {value:.4f}")

- Accuracy: 0.7740
- Precision: 0.7714
- Recall: 0.7786
- F1-Score: 0.7750


## Word2Vec + Logistic Regression

In [16]:
X_train_tokenized = [[word for word in word_tokenize(text.lower()) if word.isalnum()] for text in X_train]
X_test_tokenized = [[word for word in word_tokenize(text.lower()) if word.isalnum()] for text in X_test]

w2v_model = Word2Vec(
    X_train_tokenized, 
    vector_size=100,
    window=5,
    min_count=3,
    workers=4
)

def get_document_vector(tokens, model):
    valid_vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(valid_vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(valid_vectors, axis=0)

X_train_w2v = np.array([get_document_vector(tokens, w2v_model) for tokens in X_train_tokenized])
X_test_w2v = np.array([get_document_vector(tokens, w2v_model) for tokens in X_test_tokenized])

lr_w2v = LogisticRegression(max_iter=1000, random_state=42)
lr_w2v.fit(X_train_w2v, y_train)

preds_w2v = lr_w2v.predict(X_test_w2v)

metrics_w2v = {
    'Accuracy': accuracy_score(y_test, preds_w2v),
    'Precision': precision_score(y_test, preds_w2v),
    'Recall': recall_score(y_test, preds_w2v),
    'F1-Score': f1_score(y_test, preds_w2v)
}

for metric_name, value in metrics_w2v.items():
    print(f"- {metric_name}: {value:.4f}")

- Accuracy: 0.5673
- Precision: 0.5661
- Recall: 0.5741
- F1-Score: 0.5701


## DistilBERT

In [18]:
train_size = 3000
test_size = 1000

train_indices = np.random.choice(len(X_train), train_size, replace=False)
test_indices = np.random.choice(len(X_test), min(test_size, len(X_test)), replace=False)

X_train_sample = X_train.iloc[train_indices].reset_index(drop=True)
y_train_sample = y_train.iloc[train_indices].reset_index(drop=True)

X_test_sample = X_test.iloc[test_indices].reset_index(drop=True)
y_test_sample = y_test.iloc[test_indices].reset_index(drop=True)

train_dataset = Dataset.from_dict({'text': X_train_sample, 'label': y_train_sample})
test_dataset = Dataset.from_dict({'text': X_test_sample, 'label': y_test_sample})

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test
)

trainer.train()

predictions = trainer.predict(tokenized_test)

preds_transformer = np.argmax(predictions.predictions, axis=-1)

metrics_transformer = {
    'Accuracy': accuracy_score(y_test_sample, preds_transformer),
    'Precision': precision_score(y_test_sample, preds_transformer),
    'Recall': recall_score(y_test_sample, preds_transformer),
    'F1-Score': f1_score(y_test_sample, preds_transformer)
}

for metric_name, value in metrics_transformer.items():
    print(f"- {metric_name}: {value:.4f}")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7361.79it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Map: 100%|██████████| 1000/1000 [00:00<00:00, 9614.76 examples/s]
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `

Step,Training Loss
10,0.689240
20,0.687018
30,0.675672
40,0.673500
50,0.653541
60,0.535909
70,0.492085
80,0.469141
90,0.445092
100,0.510348


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.46it/s]
c:\Users\poyas\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


- Accuracy: 0.8310
- Precision: 0.8257
- Recall: 0.8468
- F1-Score: 0.8361


## Results

In [24]:
data = {
    'TF-IDF + LogReg': metrics_tfidf,
    'Word2Vec + LogReg': metrics_w2v,
    'DistilBERT (Fine-tuned)': metrics_transformer
}

df_results = pd.DataFrame(data).T
df_results = df_results.round(4)
df_results


,Accuracy,Precision,Recall,F1-Score
TF-IDF + LogReg,0.7740,0.7714,0.7786,0.7750
Word2Vec + LogReg,0.5673,0.5661,0.5741,0.5701
DistilBERT (Fine-tuned),0.8310,0.8257,0.8468,0.8361
